In [9]:
import cv2
import face_recognition as fr
import os
import numpy as np
import csv
from datetime import datetime

In [10]:
path = "imgs"
images = []
names = []
encodings = []

In [11]:
for file in os.listdir(path):
    img = fr.load_image_file(f"{path}/{file}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    encode = fr.face_encodings(img)[0]
    encodings.append(encode)
    names.append(os.path.splitext(file)[0])

In [12]:
def markAttendance(name):
    with open("attendance.csv", "a", newline="") as f:
        writer = csv.writer(f)
        time = datetime.now().strftime("%H:%M:%S")
        writer.writerow([name, time])

In [14]:

cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    faces = fr.face_locations(rgb)
    face_encodes = fr.face_encodings(rgb, faces)

    for encode, face in zip(face_encodes, faces):
        matches = fr.compare_faces(encodings, encode)
        dist = fr.face_distance(encodings, encode)
        idx = np.argmin(dist)

        if matches[idx]:
            name = names[idx]

            if name not in marked:
                markAttendance(name)
                marked.add(name)

            y1, x2, y2, x1 = face
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, name, (x1,y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.imshow("Attendance", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()